# 08 - GRU Metrics on DB-First Sequences

Inspect the persisted GRU state against the active DB-first build.

This notebook:
- loads the active DB-first build and imported OpenClaw rows
- reads the latest GRU weight metadata from KV
- reads live-training result metrics from `.vault-exec/live-training`
- reports dataset volume, split counts, and the latest saved evaluation metrics

It does not retrain the GRU and it does not re-run full model inference.


In [1]:
import {
  getActiveTrainingBuildId,
  listActiveSessionSequences,
  listActiveToolLeafEdgesNext,
  listActiveToolLeafNodes,
} from "../src/training-data/rebuild.ts";
import { openVaultStore } from "../src/db/index.ts";
import { OpenClawLocalStore } from "../src/ingest/local-store.ts";
import {
  readLiveTrainingFailure,
  readLiveTrainingResult,
} from "../src/service/training/result.ts";
import { resolveLiveTrainingPaths } from "../src/service/training/state.ts";

const VAULT_PATH = "../demo-vault";
const DB_PATH = `${VAULT_PATH}/.vault-exec/vault.kv`;
const MIN_CALLS = 3;
const INCLUDE_SUBAGENTS = true;

async function readOptionalLiveResult(resultDir: string) {
  try {
    return await readLiveTrainingResult(resultDir);
  } catch {
    return null;
  }
}

function countExamples(rows: Array<{ callCount: number }>) {
  return rows.reduce((sum, row) => sum + Math.max(0, row.callCount - 1), 0);
}

const kv = await Deno.openKv(DB_PATH);
const buildId = await getActiveTrainingBuildId(kv);
if (!buildId) {
  throw new Error(`No active DB-first build found in ${DB_PATH}. Run 'deno task cli init ${VAULT_PATH}' first.`);
}
const nodeRows = await listActiveToolLeafNodes(kv);
const edgeRows = await listActiveToolLeafEdgesNext(kv);
const sequenceRows = await listActiveSessionSequences(kv);
kv.close();

const openClawStore = await OpenClawLocalStore.open(DB_PATH);
const toolCallRows = await openClawStore.listToolCalls();
openClawStore.close();

const store = await openVaultStore(DB_PATH);
const latestWeights = await store.getLatestWeights();
store.close();

const retainedSequences = sequenceRows.filter((row) => row.callCount >= MIN_CALLS)
  .filter((row) => INCLUDE_SUBAGENTS || row.sessionKind !== "subagent");
const retainedSequenceRows = retainedSequences.map((row) => ({
  sessionId: row.sessionId,
  sessionKind: row.sessionKind,
  callCount: row.callCount,
}));
const topLevelSequences = retainedSequences.filter((row) => row.sessionKind === "top_level");
const subagentSequences = retainedSequences.filter((row) => row.sessionKind === "subagent");

const exampleCounts = {
  all: countExamples(retainedSequences),
  topLevel: countExamples(topLevelSequences),
  subagent: countExamples(subagentSequences),
};

const paths = resolveLiveTrainingPaths(VAULT_PATH);
const resultDir = `${paths.resultDir}/${buildId}/gru`;
const liveResult = await readOptionalLiveResult(resultDir);
const liveFailure = await readLiveTrainingFailure(resultDir);

const datasetSummary = {
  buildId,
  nodeCount: nodeRows.length,
  edgeCount: edgeRows.length,
  sessionSequenceCount: sequenceRows.length,
  retainedSequenceCount: retainedSequences.length,
  toolCallCount: toolCallRows.length,
  weightsEpoch: latestWeights?.epoch ?? null,
  weightsVocabSize: latestWeights?.vocabSize ?? null,
  weightsAccuracy: latestWeights?.accuracy ?? null,
  liveRunId: liveResult?.runId ?? null,
  liveDurationMs: liveResult?.metrics.durationMs ?? null,
  liveFailure: liveFailure?.error ?? null,
};

console.log(`Active build: ${datasetSummary.buildId}`);
console.log(`Leaf nodes: ${datasetSummary.nodeCount}`);
console.log(`Leaf transitions: ${datasetSummary.edgeCount}`);
console.log(`Session sequences: ${datasetSummary.sessionSequenceCount}`);
console.log(`Retained sequences (min_calls >= ${MIN_CALLS}): ${datasetSummary.retainedSequenceCount}`);
console.log(`Imported tool calls: ${datasetSummary.toolCallCount}`);
console.log(`Training examples (all): ${exampleCounts.all}`);
console.log(`Training examples (top-level): ${exampleCounts.topLevel}`);
console.log(`Training examples (subagent): ${exampleCounts.subagent}`);
if (latestWeights) {
  console.log(`Persisted GRU epoch: ${latestWeights.epoch}`);
  console.log(`Persisted GRU accuracy meta: ${(latestWeights.accuracy * 100).toFixed(1)}%`);
}
if (liveResult) {
  console.log(`Latest live-training run: ${liveResult.runId}`);
  console.log(`Latest live-training accuracy: ${(liveResult.metrics.accuracy * 100).toFixed(1)}%`);
  console.log(`Latest live-training top-3 accuracy: ${(liveResult.metrics.top3Accuracy * 100).toFixed(1)}%`);
  console.log(`Latest live-training MRR: ${liveResult.metrics.mrr.toFixed(4)}`);
  console.log(`Latest live-training majority baseline: ${(liveResult.metrics.majorityNextBaseline * 100).toFixed(1)}%`);
}
if (liveFailure) {
  console.log(`Latest live-training failure: ${liveFailure.error}`);
}
console.table(retainedSequenceRows.slice(0, 20));

Active build: 2b103f4f-a717-44f2-83ef-305eb0149166


Leaf nodes: 107


Leaf transitions: 994


Session sequences: 231


Retained sequences (min_calls >= 3): 221


...

Training examples (top-level): 8516


Training examples (subagent): 0


Persisted GRU epoch: 1


Persisted GRU accuracy meta: 0.0%


┌───────┬────────────────────────────────────────┬─────────────┬───────────┐
│ (idx) │ sessionId                              │ sessionKind │ callCount │
├───────┼────────────────────────────────────────┼─────────────┼───────────┤
│     0 │ "0163dfd6-f4d3-4f67-9731-e7f567769c9f" │ "top_level" │        35 │
│     1 │ "01cd4a19-4a47-43e6-8d40-5c8527bac27c" │ "top_level" │         6 │
│     2 │ "04211af8-bad4-4bad-b29b-d1ed7b7ec762" │ "top_level" │        33 │
│     3 │ "07a97ae5-c08f-482c-9250-1c5d48412cfd" │ "top_level" │         5 │
│     4 │ "08404db6-bd42-433b-8212-3a3c7e0d5fbc" │ "top_level" │         6 │
│     5 │ "0842ee73-ba8e-43fe-8975-d37e7920697e" │ "top_level" │         4 │
│     6 │ "086dc792-cfc7-489b-95c4-acab9dd6fe1f" │ "top_level" │         3 │
│     7 │ "090ee147-867a-456b-8664-6ef963353c8d" │ "top_level" │         3 │
│     8 │ "0bbf300d-7def-4df1-87ad-46a660fba6ad" │ "top_level" │         7 │
│     9 │ "0bd6e578-58a7-4763-bcaf-3bf4e39b3d2f" │ "top_level" │        44 │

In [2]:
const metricRows = liveResult
  ? [
    { metric: "accuracy", value: liveResult.metrics.accuracy },
    { metric: "top3Accuracy", value: liveResult.metrics.top3Accuracy },
    { metric: "mrr", value: liveResult.metrics.mrr },
    { metric: "majorityNextBaseline", value: liveResult.metrics.majorityNextBaseline },
  ]
  : [];

const splitRows = [
  { split: "all", examples: exampleCounts.all },
  { split: "top_level", examples: exampleCounts.topLevel },
  { split: "subagent", examples: exampleCounts.subagent },
];

if (metricRows.length > 0) {
  console.table(metricRows.map((row) => ({ ...row, value: row.value.toFixed(4) })));
} else {
  console.log("No live-training result artifact for the active build yet.");
}
console.table(splitRows);

No live-training result artifact for the active build yet.


┌───────┬─────────────┬──────────┐
│ (idx) │ split       │ examples │
├───────┼─────────────┼──────────┤
│     0 │ "all"       │     8516 │
│     1 │ "top_level" │     8516 │
│     2 │ "subagent"  │        0 │
└───────┴─────────────┴──────────┘


In [3]:
const metricsSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  title: liveResult ? "Latest GRU metrics from live training" : "GRU metrics unavailable for the active build",
  width: 760,
  height: 220,
  data: { values: metricRows },
  mark: "bar",
  encoding: {
    x: { field: "metric", type: "nominal", title: "Metric" },
    y: { field: "value", type: "quantitative", title: "Score" },
    tooltip: [
      { field: "metric" },
      { field: "value", format: ".4f" }
    ]
  }
};

const splitSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  title: "Training-example volume by split",
  width: 760,
  height: 180,
  data: { values: splitRows },
  mark: "bar",
  encoding: {
    x: { field: "split", type: "nominal", title: "Split" },
    y: { field: "examples", type: "quantitative", title: "Examples" },
    tooltip: [
      { field: "split" },
      { field: "examples" }
    ]
  }
};

const lengthSpec = {
  "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
  title: `Sequence length distribution (min_calls >= ${MIN_CALLS})`,
  width: 760,
  height: 220,
  data: { values: retainedSequenceRows },
  mark: "bar",
  encoding: {
    x: { field: "callCount", type: "quantitative", bin: true, title: "Calls per sequence" },
    y: { aggregate: "count", type: "quantitative", title: "Sequences" },
    color: { field: "sessionKind", type: "nominal", title: "Session kind" }
  }
};

({
  vconcat: [metricsSpec, splitSpec, lengthSpec],
  config: { view: { stroke: null } },
});

{
  vconcat: [
    {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      title: "GRU metrics unavailable for the active build",
      width: 760,
      height: 220,
      data: { values: [] },
      mark: "bar",
      encoding: {
        x: { field: "metric", type: "nominal", title: "Metric" },
        y: { field: "value", type: "quantitative", title: "Score" },
        tooltip: [ [Object], [Object] ]
      }
    },
    {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      title: "Training-example volume by split",
      width: 760,
      height: 180,
      data: { values: [ [Object], [Object], [Object] ] },
      mark: "bar",
      encoding: {
        x: { field: "split", type: "nominal", title: "Split" },
        y: { field: "examples", type: "quantitative", title: "Examples" },
        tooltip: [ [Object], [Object] ]
      }
    },
    {
      "$schema": "https://vega.github.io/schema/vega-lite/v5.json",
      title: "Sequence length distr